# Case by case analysis

The goal of this notebook is to analyze a sub-set of episodes that failed in the previous runs of validation.ipynb

In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import json, os
import numpy as np

# Reflect
from main.data import load_data
from main.audio import process_sound, audio2label
from main.execute_replan import run_correction_mem
from main.scene_graph import SceneGraph, Node as SceneGraphNode
from main.get_local_sg import get_scene_graph
from main.utils import get_label_from_object_id, convert_step_to_timestep, get_robot_plan_mem, get_scene_text_util, get_replan_prefix, translate_plan
from main.exp import get_scene_text
from LLM.prompt import LocalLLMPrompter
import main.constants as cons

In [ ]:
# Support functions

# Evaluation helpers 

def ts_to_sec(ts):
    """Convert 'MM:SS' timestamp string to total seconds (int)."""
    m, s = ts.strip().split(":")
    return int(m) * 60 + int(s)

def assess_error_localization(pred_list, gt_list) -> bool:
    if not pred_list or not gt_list:
        return False
    try:
        p_sec = ts_to_sec(pred_list[0])
        gt_secs = [ts_to_sec(t) for t in gt_list]
        if len(gt_secs) == 1:
            return p_sec == gt_secs[0]
        else:
            return gt_secs[0] <= p_sec <= gt_secs[-1]
    except Exception:
        return False

#  Data loading

def load_episode_data(sim_data_dir, episode_name):
    """Load all data for one episode; return (events, task_detail, object_list, interact_actions, nav_actions, data_path)."""
    task_name = episode_name.split('-')[0]
    data_path = os.path.join(sim_data_dir, task_name, episode_name)
    with open(os.path.join(data_path, "task.json"), 'r') as f:
        task_detail = json.load(f)
    events, task_detail, object_list, interact_actions, nav_actions = load_data(data_path, task_detail)
    return events, task_detail, object_list, interact_actions, nav_actions, data_path

def detect_episode_sounds(data_path, object_list):
    """Return detected sounds dict with 1-based step keys."""
    return {k + 1: v for k, v in process_sound(data_path, object_list).items()}

# Scene graph construction

def build_episode_scene_graphs(events, object_list, interact_actions, nav_actions, task_detail, detected_sounds):
    """Build global and per-step local scene graphs for an episode.

    Returns (global_sg, key_frames, local_graphs, object_list_set).
    """
    global_sg = SceneGraph(events[-1], task_detail)
    key_frames = set()
    local_graphs = {}
    prev_graph = SceneGraph(event=None, task=task_detail)
    total_points_dict, bbox3d_dict = {}, {}
    obj_held_prev = None
    count, interval = 0, 2
    action_frames = set(interact_actions) | set({idx[1] for idx in nav_actions.keys()})
    keyframe_triggers = action_frames | set(detected_sounds)

    for step_id, event in enumerate(events, 1):
        if step_id not in action_frames:
            count += 1
            if step_id not in detected_sounds and count % interval == 0:
                continue
        local_sg, total_points_dict, obj_held_prev, bbox3d_dict = get_scene_graph(
            step_id - 1, event, object_list, total_points_dict, bbox3d_dict, obj_held_prev, task_detail
        )
        local_graphs[step_id] = local_sg
        if local_sg != prev_graph:
            key_frames.add(step_id)
            prev_graph = local_sg
        if step_id in keyframe_triggers:
            key_frames.add(step_id)

    label_names = {label: get_label_from_object_id(label, events, task_detail) for label in total_points_dict}
    object_list_set = set(object_list)

    for label, name in label_names.items():
        if name is not None:
            global_sg.add_node_wo_edge(SceneGraphNode(
                name=name, object_id=label,
                pos3d=bbox3d_dict[label].get_center(),
                corner_pts=np.array(bbox3d_dict[label].get_box_points()),
                pcd=total_points_dict[label], global_node=True
            ))

    node_by_name = {node.name: node for node in global_sg.total_nodes}
    for label, name in label_names.items():
        if label.split("|")[0] in object_list_set and name in node_by_name:
            global_sg.add_node(node_by_name[name])

    global_sg.add_agent()
    return global_sg, key_frames, local_graphs, object_list_set

# Caption building

def _get_held_object(step_id, local_graphs):
    """Walk backwards until we find a local_sg that has a held object."""
    while step_id >= 0:
        sg = local_graphs.get(step_id)
        if sg is not None:
            for key in sg.edges:
                if "robot gripper" in key and key[0] != "nothing":
                    return key[0]
        step_id -= 1
    return None

def _find_action_for_step(step_id_1based, interact_actions, nav_actions):
    """Helper to find action for a given step."""
    if step_id_1based in interact_actions:
        return interact_actions[step_id_1based], True
    for (min_step, max_step), action in nav_actions.items():
        if min_step <= step_id_1based <= max_step:
            return action, False
    return None, None

def build_episode_captions(events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds):
    """Build L1 (step-level) and L2 (subgoal-level) caption lists.

    Returns (L1_captions, L2_captions, state_summary_L1, state_summary_L2).
    L1_captions is a list of (step_id, caption_str) tuples.
    """
    L1_captions = []

    for step_id, _ in enumerate(events):
        if step_id + 1 not in local_graphs or step_id + 1 not in key_frames:
            continue
        action, _ = _find_action_for_step(step_id + 1, interact_actions, nav_actions)
        if not action:
            continue
        caption = f"{convert_step_to_timestep(step_id + 1, 1)}. Action: {action}. "
        caption += f"Visual observation: {get_scene_text(local_graphs[step_id + 1])}"
        # Use model-detected sounds (1-based keys) to match the validation pipeline
        if (step_id + 1) in detected_sounds:
            caption += f" Auditory observation: {detected_sounds[step_id + 1]}."
        caption += "\n"
        L1_captions.append((step_id + 1, caption))

    L2_captions = [cap.replace("Action", "Goal") for step_id, cap in L1_captions if step_id in interact_actions]
    state_summary_L1 = "".join(cap for _, cap in L1_captions)
    state_summary_L2 = "".join(L2_captions)
    return L1_captions, L2_captions, state_summary_L1, state_summary_L2

# LLM prompt utility

def build_prompt(info, **replacements):
    """Construct a {system, user} prompt dict from a template, applying keyword substitutions."""
    user_text = info['template-user']
    for key, value in replacements.items():
        user_text = user_text.replace(f"[{key.upper()}]", value)
    return {'system': info['template-system'], 'user': user_text}

# Reasoning pipeline

def run_reasoning(L1_captions, L2_captions, task, global_sg, llm_prompter, prompt_template):
    """Run subgoal verification → failure reasoning → GT labelling.

    Returns reasoning_dict.
    """
    L1_captions_text = [cap for _, cap in L1_captions]
    sv_info = prompt_template['subgoal-verifier']
    failed_caption = None

    for caption in L2_captions:
        subgoal = caption.split(". ")[1].split(": ")[1].lower()
        observation = caption[caption.find("Visual observation"):]
        prompt = build_prompt(sv_info, subgoal=subgoal, observation=observation)
        ans, _ = llm_prompter.query(prompt, sampling_params=sv_info['params'])
        if ans.split(", ")[0] != "Yes":
            failed_caption = caption
            break

    reasoning_dict = {}

    if failed_caption:  # execution error
        step_name = failed_caption.split(".")[0]
        for step_id, caption in L1_captions:
            if step_name not in caption:
                continue
            action = caption.split(". ")[1].split(": ")[1].lower()
            prev_observations = get_robot_plan_mem(L2_captions, L1_captions_text, step=step_name, with_obs=True)
            observation = caption[caption.find("Action"):]
            prompt_key = 'reasoning-execution' if prev_observations else 'reasoning-execution-no-history'
            re_info = prompt_template[prompt_key]
            re_prompt = build_prompt(re_info, action=action, task_name=task['name'],
                                     step=step_name, summary=prev_observations, observation=observation)
            failure_reason, _ = llm_prompter.query(prompt=re_prompt, sampling_params=re_info['params'])
            res_info = prompt_template['reasoning-execution-steps']
            res_prompt = build_prompt(res_info, failure_reason=failure_reason)
            time_steps, _ = llm_prompter.query(prompt=res_prompt, sampling_params=res_info['params'])
            reasoning_dict.update({
                'pred_failure_reason': failure_reason,
                'pred_failure_step':   [ts.strip(",") for ts in time_steps.split(", ")],
                'error_type':          'execution',
            })
            break

    else:  # planning error
        observation = get_robot_plan_mem(L2_captions, L1_captions_text, step=None, with_obs=False)
        rp_info = prompt_template['reasoning-plan']
        rp_prompt = build_prompt(rp_info, task_name=task['name'],
                                 success_condition=task['success_condition'],
                                 current_state=get_scene_text_util(global_sg),
                                 observation=observation)
        failure_reason, _ = llm_prompter.query(prompt=rp_prompt, sampling_params=rp_info['params'])
        rps_info = prompt_template['reasoning-plan-steps']
        rps_prompt = build_prompt(rps_info, prev_prompt=rp_prompt['user'] + " " + failure_reason)
        failure_step, _ = llm_prompter.query(prompt=rps_prompt, sampling_params=rps_info['params'])
        reasoning_dict.update({
            'pred_failure_reason': failure_reason,
            'pred_failure_step':   [failure_step.split(" ")[0].rstrip('.,')],
            'error_type':          'planning',
        })

    reasoning_dict.update({
        'gt_failure_reason': task['gt_failure_reason'],
        'gt_failure_step':   task['gt_failure_step'],
        'gt_error_type':     'planning' if task.get('chosen_failure') else 'execution',
    })
    return reasoning_dict

# Replanning

def get_initial_plan(actions):
    """Format the original plan, excluding navigation actions."""
    plan = ""
    idx = 0
    for action in actions:
        params = action[1:-1].split(", ")
        if params[0] != "navigate_to_obj":
            for i, obj_name in enumerate(params[1:]):
                params[i + 1] = cons.NAME_MAP.get(obj_name, obj_name.lower())
            plan += f"{idx + 1}. ({', '.join(params)})\n"
            idx += 1
    return plan[:-1]

def run_replanning(task, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template):
    """Generate a corrected plan given the failure reasoning.

    Returns replan_dict with keys: task_plan, llm_plan_raw, llm_plan, num_steps.
    """
    last_event = events[-1]
    current_state = get_scene_text_util(global_sg)
    global_object_list = list(
        set([obj["objectType"] for obj in last_event.metadata["objects"]]) | object_list_set
    )
    corr_info = prompt_template['correction']
    prompt = {
        'system': corr_info['template-system'].replace("[PREFIX]", get_replan_prefix()),
        'user': (
            corr_info['template-user']
            .replace("[TASK_NAME]", task['name'])
            .replace("[PLAN]", get_initial_plan(task['actions']))
            .replace("[FAILURE_REASON]", reasoning_dict['pred_failure_reason'])
            .replace("[CURRENT_STATE]", current_state)
            .replace("[SUCCESS_CONDITION]", task['success_condition'])
        ),
    }
    plan, _ = llm_prompter.query(prompt=prompt, sampling_params=corr_info['params'])
    translated_plan = translate_plan(plan, global_object_list, last_event)
    translated_lines = translated_plan.split("\n")[:-1]
    return {
        'task_plan': list(task['actions']),
        'llm_plan_raw': plan,
        'llm_plan': translated_lines,
        'num_steps': len(translated_lines),
    }

# Correction

def run_episode_correction(episode_name, task_name, data_path, task_detail, events, reasoning_dict, replan_dict):
    """Save reasoning and run the correction; returns correction_dict."""
    last_frame = len(events) - 1
    _reasoning_dir = os.path.join(data_path, 'state_summary', task_name, episode_name)
    os.makedirs(_reasoning_dir, exist_ok=True)
    with open(os.path.join(_reasoning_dir, 'reasoning.json'), 'w') as _rf:
        json.dump(reasoning_dict, _rf, default=str)
    return run_correction_mem(data_path, task_detail, events[-1], last_frame, replan_dict)

# Analysis output

def print_episode_analysis(episode_name, task, reasoning_dict, replan_dict):
    """Print a side-by-side comparison of GT vs. predicted reasoning and the correction plan."""
    print(f"Episode name: {episode_name}")
    print(f"Original task: {task['name']}")
    print(f"Original failure reason: {reasoning_dict['gt_failure_reason']}")
    print(f"Predicted failure reason: {reasoning_dict['pred_failure_reason']}")
    print(f"GT failure step(s): {reasoning_dict['gt_failure_step']}")
    print(f"Predicted failure step(s): {reasoning_dict['pred_failure_step']}")
    print(f"Side to side plans:\nOriginal plan:\n{replan_dict['llm_plan_raw']}")
    print("Correction plan:")
    for line, instruct in enumerate(replan_dict['llm_plan'], 1):
        print(f"{line}. {instruct}")

# Constants
SIM_DATA_DIR = "./datasets/sim_data/"


In [ ]:
# Get previous run data
from reflect.core.paths import sim_results_root

RESULTS_PATH = str(sim_results_root()) + "/"
PREVIOUS_RUN_PATH = os.path.join(RESULTS_PATH, "reflect_results_run1.json")

with open(PREVIOUS_RUN_PATH, 'r') as f:
    previous_run_data = json.load(f)
previous_run_data = pd.DataFrame(previous_run_data)

# Get human evaluation scores
EVAL_CSV_PATH = os.path.join(RESULTS_PATH, "exp_evaluation_run1.csv")
scored = pd.read_csv(EVAL_CSV_PATH)

# Join human scores with previous run data
previous_run_data = previous_run_data.merge(scored[['episode', 'human_score']], on='episode', how='left')

# Unfold dict columns for easier analysis
unfolded_reasoning = pd.json_normalize(previous_run_data['reasoning_dict'])
unfolded_replanning = pd.json_normalize(previous_run_data['replan_dict'])
unfolded_corrections = pd.json_normalize(previous_run_data['correction_dict'])

# Combine all data into a single DataFrame
run_df = pd.concat([previous_run_data.drop(columns=['reasoning_dict', 'replan_dict', 'correction_dict']),
                    unfolded_reasoning.add_prefix('reasoning_'),
                    unfolded_replanning.add_prefix('replan_'),
                    unfolded_corrections.add_prefix('correction_')], axis=1)

run_df.head()


In [ ]:
# Filter to failed episodes and relevant cases by error degree
failed_episodes = run_df[run_df['correction_success'] == False].copy()

# Compute metrics:
# Exp (explanation): Either False or True, False if failure explanations is not correct and informative or True if it is. Determined by human evaluators.
# Loc (localization): Either False or True, False if the error is not correctly located in the reasoning trace, or True if it is. Determined by whether the located error step matches
# the precise step if gt step = 1 in dataset or within range if gt step is a range in dataset.
# Co-plan (correction planning success rate): Either False or True, False if the correction plan did not successfully fix the error and lead to a successful episode, or True if it did. Determined by whether the episode is successful after applying the correction plan.
failed_episodes.rename(columns={'correction_success': 'Co-plan'}, inplace=True)
failed_episodes['Loc'] = failed_episodes.apply(lambda row: assess_error_localization(row['reasoning_pred_failure_step'], row['reasoning_gt_failure_step']), axis=1)
failed_episodes.rename(columns={'human_score': 'Exp'}, inplace=True)
failed_episodes['Exp'] = failed_episodes['Exp'].astype(bool)

# Group 1: Error not located and not correctly explained
group1 = failed_episodes[(failed_episodes['Loc'] == False) & (failed_episodes['Exp'] == False)]

# Group 2: Error located but not correctly explained
group2 = failed_episodes[(failed_episodes['Loc'] == True) & (failed_episodes['Exp'] == False)]

# Group 3: Error located and correctly explained
group3 = failed_episodes[(failed_episodes['Loc'] == True) & (failed_episodes['Exp'] == True)]

# Group 4: Error not located but correctly explained
group4 = failed_episodes[(failed_episodes['Loc'] == False) & (failed_episodes['Exp'] == True)]

print(f"Total failed episodes: {len(failed_episodes)}")
print(f"Total failed episodes after grouping: {len(group1) + len(group2) + len(group3) + len(group4)}")

In [ ]:
plt.figure(figsize=(14, 8))
plt.bar('Group 1: Not Located & Not Explained', len(group1), color='#4c72b0')
plt.bar('Group 2: Located & Not Explained', len(group2), color='#e07b39')
plt.bar('Group 3: Located & Explained', len(group3), color='#55a868')
plt.bar('Group 4: Not Located & Explained', len(group4), color='#c44e52')
plt.title('Distribution of Failed Episodes by Explanation and Localization Quality', fontsize=14, fontweight='bold')
plt.ylabel('Number of Failed Episodes', fontsize=12)
plt.legend()
plt.show()

## Distribution

In [ ]:
# In total we're going to analyze one by one 2 episodes from each group, so 8 episodes in total. 
# We will analyze the reasoning traces, the predicted and gt failure steps, the explanations, and the correction plans for these episodes 
# to understand the patterns of failure and how they relate to explanation quality, localization accuracy, and correction success.
sampled_episodes = {
    'g1': group1.sample(2, random_state=42)['episode'].tolist(),
    'g2': group2.sample(2, random_state=42)['episode'].tolist(),
    'g3': group3.sample(2, random_state=42)['episode'].tolist(),
    'g4': group4.sample(2, random_state=42)['episode'].tolist(),
}

print("Sampled Episodes for Analysis:")
for group, episodes in sampled_episodes.items():
    print(f"{group}: {episodes}")

In [ ]:
# LLM setup (shared across all episodes)
LLM_DIR     = "./LLM"
LOCAL_MODEL = "qwen3.5:9b"
OLLAMA_URL  = "http://localhost:11434"

llm_prompter = LocalLLMPrompter(model_name=LOCAL_MODEL, base_url=OLLAMA_URL)
with open(os.path.join(LLM_DIR, "prompts.json")) as f:
    prompt_template = json.load(f)

## Group 1

### Episode 1

In [ ]:
# Load and process episode
episode_name = sampled_episodes['g1'][0]
task_name = episode_name.split('-')[0]
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode_name)
task = task_detail

detected_sounds = detect_episode_sounds(data_path, object_list)
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(
    events, object_list, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions, L2_captions, _, _ = build_episode_captions(
    events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions_text = [cap for _, cap in L1_captions]

print(f"Loaded episode: {episode_name}")
print(f"Events: {len(events)}, Objects: {len(object_list)}, Interact actions: {len(interact_actions)}, Nav actions: {len(nav_actions)}")
print(f"Scene graph: {len(global_sg.total_nodes)} nodes, {len(global_sg.edges)} edges")
print(f"Detected sounds: {detected_sounds}")

In [ ]:
# Reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task, global_sg, llm_prompter, prompt_template)
print("Reasoning dict for the episode:")
print(json.dumps(reasoning_dict, indent=4))

In [ ]:
# Replanning
replan_dict = run_replanning(task, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print("Replanning dict for the episode:")
print(json.dumps(replan_dict, indent=4))

In [ ]:
# Run correction
correction_dict = run_episode_correction(episode_name, task_name, data_path, task_detail, events, reasoning_dict, replan_dict)
print("Correction dict for the episode:")
print(json.dumps(correction_dict, indent=4))

#### Error analysis

In [ ]:
print_episode_analysis(episode_name, task, reasoning_dict, replan_dict)

The LLM was able to depict the moment of error but not why the error happened exactly. Why didn't it mention the bread? In order to answer this, let's inspect what the LLM saw:

In [ ]:
for cap in L1_captions_text:
    print(cap)

It seems that the bread is not mentioned here either, let's look at the actual scene graph.

In [ ]:
print(f"Scene graph at failure point: {local_graphs[7]}")

It seems that the visual observation excluded the element altogether. Why?

In [ ]:
print(object_list)

After reviewing the original code for computing local scene graphs, we have observed that a filter to only add nodes relevant to the task is present. This object list is constructed form a function that extracts objects mentioned in the original plan.

*Conclusion*: The problem comes from filtering nodes in graph computing to objects present in the original plan. This seems to be more of a problem with the specific implementation rather than the pipeline. As this is not mentioned in the original paper and it's more of an efficiency optimization.

*Possible fix*: Remove object filtering and thus, add more detail to the observations.

### Episode 2

In [ ]:
# Load and process episode
episode_name = sampled_episodes['g1'][1]
task_name = episode_name.split('-')[0]
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode_name)
task = task_detail

detected_sounds = detect_episode_sounds(data_path, object_list)
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(
    events, object_list, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions, L2_captions, _, _ = build_episode_captions(
    events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions_text = [cap for _, cap in L1_captions]

print(f"Loaded episode: {episode_name}")
print(f"Events: {len(events)}, Objects: {len(object_list)}, Interact actions: {len(interact_actions)}, Nav actions: {len(nav_actions)}")
print(f"Scene graph: {len(global_sg.total_nodes)} nodes, {len(global_sg.edges)} edges")
print(f"Detected sounds: {detected_sounds}")

In [ ]:
# Reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task, global_sg, llm_prompter, prompt_template)
print("Reasoning dict for the episode:")
print(json.dumps(reasoning_dict, indent=4))

In [ ]:
# Replanning
replan_dict = run_replanning(task, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print("Replanning dict for the episode:")
print(json.dumps(replan_dict, indent=4))

In [ ]:
# Run correction
correction_dict = run_episode_correction(episode_name, task_name, data_path, task_detail, events, reasoning_dict, replan_dict)
print("Correction dict for the episode:")
print(json.dumps(correction_dict, indent=4))

#### Error analysis

In [ ]:
print_episode_analysis(episode_name, task, reasoning_dict, replan_dict)

It seems that the LLM found an error where there wasn't supposed to be one, let's see what the LLM observed:

In [ ]:
print(L1_captions_text[35])
print(local_graphs[150]) # same step as predicted failure step

*Conclusion*: After reviewing the actual video it seems that the problem was the method to compute inter-object relationships. While the positioning was a bit weird because of the simulation, the lettuce was in fact inside the bowl and not on top as indicated by the edge. So the LLM did assess the situation correctly on an incorrect context.

*Possible fix*: Transition to a more robust perception system that can account to a wider range of object positioning and interpretation.

## Group 2

### Episode 1

In [ ]:
# Load and process episode
episode_name = sampled_episodes['g2'][0]
task_name = episode_name.split('-')[0]
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode_name)
task = task_detail

detected_sounds = detect_episode_sounds(data_path, object_list)
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(
    events, object_list, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions, L2_captions, _, _ = build_episode_captions(
    events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions_text = [cap for _, cap in L1_captions]

print(f"Loaded episode: {episode_name}")
print(f"Events: {len(events)}, Objects: {len(object_list)}, Interact actions: {len(interact_actions)}, Nav actions: {len(nav_actions)}")
print(f"Scene graph: {len(global_sg.total_nodes)} nodes, {len(global_sg.edges)} edges")
print(f"Detected sounds: {detected_sounds}")

In [ ]:
# Reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task, global_sg, llm_prompter, prompt_template)
print("Reasoning dict for the episode:")
print(json.dumps(reasoning_dict, indent=4))

In [ ]:
# Replanning
replan_dict = run_replanning(task, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print("Replanning dict for the episode:")
print(json.dumps(replan_dict, indent=4))

In [ ]:
# Run correction
correction_dict = run_episode_correction(episode_name, task_name, data_path, task_detail, events, reasoning_dict, replan_dict)
print("Correction dict for the episode:")
print(json.dumps(correction_dict, indent=4))

#### Error analysis

In [ ]:
print_episode_analysis(episode_name, task, reasoning_dict, replan_dict)

There are multiple errors in this example, the explanation does explain in some way the elements that where involved in the error but not the actual error. The second problem with this episode is that although the error localization wasn't entirely correct the plan should have worked but failed to correctly translate the LLM plan to the primitives:

5. (pour, pot, house plant) => (pour, HousePlant, Pot) score: 0.9812078

First let's explore what the LLM saw to see why the explanation quality is of such poor quality.

In [ ]:
for cap in L1_captions_text:
    print(cap)
print(local_graphs[34]) # same step as predicted failure step

In this episode, I believe the LLM had sufficient information to infer what the problem was, this could be a problem of model thinking/size.

Now, let's explore why the plan mathcing failed for this specific episode.

In [ ]:
replan_dict = run_replanning(task, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)

After following the memory trace and execution, we can determine that the problem is entirely of the model that evaluates the labels, both options existed, with the correct and incorrect order. The option with incorrect order got a score of 0.981 and the option with the correct option got a 0,979.

*Conclusion*: The LLM model correctly identified where the error happened but did not correctly identify why, providing a very ambiguous failure explanation. The cosine similarity comparison to translate the plan, got a better similarity with the primitive action in the wrong order. This can be explained by understanding that the sentence transformer was trained on natural language, not symbolic predicates, thus, it cannot asses that argument position is important in this case.

*Possible fix*: Try using qwen in thinking mode or using a bigger model. Use a sentence transformer trained on symbolic predicates. Possibly let the llm output the sctrutured output directly.

### Episode 2

In [ ]:
# Load and process episode
episode_name = sampled_episodes['g2'][1]
task_name = episode_name.split('-')[0]
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode_name)
task = task_detail

detected_sounds = detect_episode_sounds(data_path, object_list)
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(
    events, object_list, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions, L2_captions, _, _ = build_episode_captions(
    events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions_text = [cap for _, cap in L1_captions]

print(f"Loaded episode: {episode_name}")
print(f"Events: {len(events)}, Objects: {len(object_list)}, Interact actions: {len(interact_actions)}, Nav actions: {len(nav_actions)}")
print(f"Scene graph: {len(global_sg.total_nodes)} nodes, {len(global_sg.edges)} edges")
print(f"Detected sounds: {detected_sounds}")

In [ ]:
# Reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task, global_sg, llm_prompter, prompt_template)
print("Reasoning dict for the episode:")
print(json.dumps(reasoning_dict, indent=4))

In [ ]:
# Replanning
replan_dict = run_replanning(task, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print("Replanning dict for the episode:")
print(json.dumps(replan_dict, indent=4))

In [ ]:
# Run correction
correction_dict = run_episode_correction(episode_name, task_name, data_path, task_detail, events, reasoning_dict, replan_dict)
print("Correction dict for the episode:")
print(json.dumps(correction_dict, indent=4))

#### Error analysis

In [ ]:
print_episode_analysis(episode_name, task, reasoning_dict, replan_dict)

*Conclusion*: After reviewing the video and object metadata, it seems that the task was completed correctly but marked as not succesful because although the steps where correctly executed, the simulator did not mark the water inside the cup as hot. This is a False Negative

*Possible Solution*: Change simulation environment.

## Group 3

### Episode 1

In [ ]:
# Load and process episode
episode_name = sampled_episodes['g3'][0]
task_name = episode_name.split('-')[0]
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode_name)
task = task_detail

detected_sounds = detect_episode_sounds(data_path, object_list)
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(
    events, object_list, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions, L2_captions, _, _ = build_episode_captions(
    events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions_text = [cap for _, cap in L1_captions]

print(f"Loaded episode: {episode_name}")
print(f"Events: {len(events)}, Objects: {len(object_list)}, Interact actions: {len(interact_actions)}, Nav actions: {len(nav_actions)}")
print(f"Scene graph: {len(global_sg.total_nodes)} nodes, {len(global_sg.edges)} edges")
print(f"Detected sounds: {detected_sounds}")

In [ ]:
# Reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task, global_sg, llm_prompter, prompt_template)
print("Reasoning dict for the episode:")
print(json.dumps(reasoning_dict, indent=4))

In [ ]:
# Replanning
replan_dict = run_replanning(task, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print("Replanning dict for the episode:")
print(json.dumps(replan_dict, indent=4))

In [ ]:
# Run correction
correction_dict = run_episode_correction(episode_name, task_name, data_path, task_detail, events, reasoning_dict, replan_dict)
print("Correction dict for the episode:")
print(json.dumps(correction_dict, indent=4))

#### Error analysis

In [ ]:
print_episode_analysis(episode_name, task, reasoning_dict, replan_dict)

In [ ]:
for cap in L1_captions_text:
    print(cap)

print(local_graphs[102])

*Conclusion*: Although the LLM had enough context, it was unable to correctly assess the failure cause and later produce a correct re-plan.

*Possible Solution*: Increase model size or activate thinking mode with a bigger context window.

### Episode 2

In [ ]:
# Load and process episode
episode_name = sampled_episodes['g3'][1]
task_name = episode_name.split('-')[0]
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode_name)
task = task_detail

detected_sounds = detect_episode_sounds(data_path, object_list)
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(
    events, object_list, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions, L2_captions, _, _ = build_episode_captions(
    events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions_text = [cap for _, cap in L1_captions]

print(f"Loaded episode: {episode_name}")
print(f"Events: {len(events)}, Objects: {len(object_list)}, Interact actions: {len(interact_actions)}, Nav actions: {len(nav_actions)}")
print(f"Scene graph: {len(global_sg.total_nodes)} nodes, {len(global_sg.edges)} edges")
print(f"Detected sounds: {detected_sounds}")

In [ ]:
# Reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task, global_sg, llm_prompter, prompt_template)
print("Reasoning dict for the episode:")
print(json.dumps(reasoning_dict, indent=4))

In [ ]:
# Replanning
replan_dict = run_replanning(task, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print("Replanning dict for the episode:")
print(json.dumps(replan_dict, indent=4))

In [ ]:
# Run correction
correction_dict = run_episode_correction(episode_name, task_name, data_path, task_detail, events, reasoning_dict, replan_dict)
print("Correction dict for the episode:")
print(json.dumps(correction_dict, indent=4))

#### Error analysis

In [ ]:
print_episode_analysis(episode_name, task, reasoning_dict, replan_dict)

In [ ]:
for cap in L1_captions_text:
    print(cap)

*Conclusion*: The problem in this episode was that the pipeline did not consider that subsequent key frames can help the LLM understand and reason about what happened. This caused the LLM to perform evaluations in a constrained window that did not provide it with sufficient context.

*Possible Solution*: Change methodology to expand context provided to the LLM. Evaluate wether providing a range can help the LLM infer a better failure reason.

## Group 4

### Episode 1

In [ ]:
# Load and process episode
episode_name = sampled_episodes['g4'][0]
task_name = episode_name.split('-')[0]
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode_name)
task = task_detail

detected_sounds = detect_episode_sounds(data_path, object_list)
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(
    events, object_list, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions, L2_captions, _, _ = build_episode_captions(
    events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions_text = [cap for _, cap in L1_captions]

print(f"Loaded episode: {episode_name}")
print(f"Events: {len(events)}, Objects: {len(object_list)}, Interact actions: {len(interact_actions)}, Nav actions: {len(nav_actions)}")
print(f"Scene graph: {len(global_sg.total_nodes)} nodes, {len(global_sg.edges)} edges")
print(f"Detected sounds: {detected_sounds}")

In [ ]:
# Reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task, global_sg, llm_prompter, prompt_template)
print("Reasoning dict for the episode:")
print(json.dumps(reasoning_dict, indent=4))

In [ ]:
# Replanning
replan_dict = run_replanning(task, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print("Replanning dict for the episode:")
print(json.dumps(replan_dict, indent=4))

In [ ]:
# Run correction
correction_dict = run_episode_correction(episode_name, task_name, data_path, task_detail, events, reasoning_dict, replan_dict)
print("Correction dict for the episode:")
print(json.dumps(correction_dict, indent=4))

#### Error analysis

In [ ]:
print_episode_analysis(episode_name, task, reasoning_dict, replan_dict)

In [ ]:
for cap in L1_captions_text:
    print(cap)

*Conclusion*: The LLM did not account for the bread being on top of the toaster, the problem was that it wasn't able to reason about that specific detail.

*Possible solution*: Increase model size or activate thinking mode.

### Episode 2

In [ ]:
# Load and process episode
episode_name = sampled_episodes['g4'][1]
task_name = episode_name.split('-')[0]
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode_name)
task = task_detail

detected_sounds = detect_episode_sounds(data_path, object_list)
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(
    events, object_list, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions, L2_captions, _, _ = build_episode_captions(
    events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions_text = [cap for _, cap in L1_captions]

print(f"Loaded episode: {episode_name}")
print(f"Events: {len(events)}, Objects: {len(object_list)}, Interact actions: {len(interact_actions)}, Nav actions: {len(nav_actions)}")
print(f"Scene graph: {len(global_sg.total_nodes)} nodes, {len(global_sg.edges)} edges")
print(f"Detected sounds: {detected_sounds}")

In [ ]:
# Reasoning
reasoning_dict = run_reasoning(L1_captions, L2_captions, task, global_sg, llm_prompter, prompt_template)
print("Reasoning dict for the episode:")
print(json.dumps(reasoning_dict, indent=4))

In [ ]:
# Replanning
replan_dict = run_replanning(task, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
print("Replanning dict for the episode:")
print(json.dumps(replan_dict, indent=4))

In [ ]:
# Run correction
correction_dict = run_episode_correction(episode_name, task_name, data_path, task_detail, events, reasoning_dict, replan_dict)
print("Correction dict for the episode:")
print(json.dumps(correction_dict, indent=4))

#### Error analysis

In [ ]:
print_episode_analysis(episode_name, task, reasoning_dict, replan_dict)

In [ ]:
for cap in L1_captions_text:
    print(cap)

*Conclusion*: The LLM correctly atributed the error but did not account for moving the bread to a cutting station befor attempting to cut it and place slice in the toaster. This can be attributed to a poor LLM reasoning capability.

*Possible solution*: Increase LLM size or activate thinking mode for qwen.

In [ ]:
# Run entire pipeline for cookEgg-2
# Load and process episode
episode_name = "heatPotato-4"
task_name = episode_name.split('-')[0]
events, task_detail, object_list, interact_actions, nav_actions, data_path = load_episode_data(SIM_DATA_DIR, episode_name)
task = task_detail
detected_sounds = detect_episode_sounds(data_path, object_list)
global_sg, key_frames, local_graphs, object_list_set = build_episode_scene_graphs(
    events, object_list, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions, L2_captions, _, _ = build_episode_captions(
    events, local_graphs, key_frames, interact_actions, nav_actions, task_detail, detected_sounds
)
L1_captions_text = [cap for _, cap in L1_captions]
reasoning_dict = run_reasoning(L1_captions, L2_captions, task, global_sg, llm_prompter, prompt_template)
replan_dict = run_replanning(task, reasoning_dict, global_sg, events, object_list_set, llm_prompter, prompt_template)
correction_dict = run_episode_correction(episode_name, task_name, data_path, task_detail, events, reasoning_dict, replan_dict)
print_episode_analysis(episode_name, task, reasoning_dict, replan_dict)